# RAFT-UP downsampling sensitivity: three real-data pairs

Reuse the 17 completed real-data settings from results_single_seed_v1. Run/resume ONLY
Deep-STARmap section_13 / section_14 at 10% (822 sampled points per side), with no manual
upper point cap. Existing successful supplemental results are reused. Failed supplemental
results are retried. Original results are never overwritten.

The merged tables preserve each result's original protocol hash and source path. The
supplement uses the current source; it is not presented as part of the old frozen protocol.
Frozen input identities and feature-cost provenance are checked before supplementation.

Accuracy: LAA and GPR with coverage A/B. Runtime: alignment and stage times. Stability:
correspondence sensitivity across ratios, not repeated-seed uncertainty. Only one pair per
family is evaluated. Set REUSE_EXISTING_RESULTS=True only to inspect the old study.


In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, platform, subprocess, traceback, itertools, gc
ROOT = Path(os.environ.get("RAFTUP_EXPERIMENT_ROOT", "/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828"))
REPRO = Path(os.environ.get("RAFTUP_REPRODUCE_ROOT", "/Users/salovjade/Library/CloudStorage/Dropbox/RAFTUP_after0320/raftup_reproduce"))
SRC = Path(os.environ.get("RAFTUP_SRC", "/Users/salovjade/Library/CloudStorage/Dropbox/boatwhere/raftup_repo/src"))
REUSE_EXISTING_RESULTS = False  # Run/resume the new protocol; True is ONLY for viewing archived results
OUT = ROOT / "all_exp/downsampling_sensitivity" / ("results_single_seed_v1" if REUSE_EXISTING_RESULTS else "results_three_real_no_cap_v2")
REPORT_OUT = OUT / "real_data_analysis"
REPORT_OUT.mkdir(parents=True, exist_ok=True)
print("Mode:", "ARCHIVED RESULTS (old 800-point cap; no new alignments)" if REUSE_EXISTING_RESULTS else "RUN/RESUME: manual sampling without an upper cap")
print("Results directory:", OUT)
ARCHIVE_OUT = ROOT / "all_exp/downsampling_sensitivity/results_single_seed_v1"
SUPPLEMENT_CASE = "deep_starmap_13_14"
SUPPLEMENT_RATIO = 0.10
PYTHON = sys.executable
RATIOS = [0.02, 0.04, 0.05, 0.067, 0.08, 0.10]
FIXED_SEED = 42
SEEDS = [FIXED_SEED]  # Exactly one run per case/ratio
REFERENCE_RATIO = 0.05
THREADS = 4
TIMEOUT_SECONDS = 3600  # per run; a timeout is recorded, never retried with smaller data
MIN_POINTS, MAX_POINTS = 2, None  # No hard upper cap; print a notice above 800 points
RETRY_FAILED = True
# Existing cached SCAN-IT cost, frozen for the entire sweep.
DEEP_COST = ROOT / "2_3d_reconstruction_exp/Deep-STARmap/raftup_pair_test_rotated/section_13__section_14/feature_cost.npy"
# Set to True only if no existing cost is available. Computed ONCE, outside alignment timing.
COMPUTE_MISSING_DEEP_COST = False
os.environ["OMP_NUM_THREADS"] = str(THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(THREADS)
os.environ["MKL_NUM_THREADS"] = str(THREADS)
os.environ.setdefault("NUMBA_CACHE_DIR", "/private/tmp/raftup_cutoff_numba" if sys.platform=="darwin" else "/tmp/raftup_cutoff_numba")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/raftup_cutoff_mpl" if sys.platform=="darwin" else "/tmp/raftup_cutoff_mpl")
sys.path.insert(0,str(SRC))
import numpy as np
import pandas as pd
import anndata as ad
import torch
from scipy import sparse
from scipy.spatial import cKDTree
from IPython.display import display
from raftup import _load_data, _cutoff_selection as cs
from raftup._cutoff_selection import run_manual_alignment
assert Path(cs.__file__).resolve() == (SRC/"raftup/_cutoff_selection.py").resolve()
torch.set_num_threads(THREADS)
OUT.mkdir(parents=True,exist_ok=True)

def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(8*1024*1024),b""): h.update(block)
    return h.hexdigest()

def write_json(path,obj):
    path=Path(path); tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False)); tmp.replace(path)

def fixed(df,ds,rf,rs,k):
    return dict(feature_cutoff_downsample=df,spatial_cutoff_downsample_um=ds,
                feature_cutoff_recovery=rf,spatial_cutoff_recovery_um=rs,k1=k,k2=k)

CASES = {
 "dlpfc_0708":dict(technology="dlpfc",pixels=138,label="original_clusters",radius=300,
    parameters=fixed(.10,400,.20,200,20),basis="historical final 0708 template"),
 "merfish_0409":dict(technology="merfish",pixels=100,label="original_clusters",radius=10,
    parameters=fixed(.20,100,.25,60,10),basis="historical final 0409 template"),
 "deep_starmap_13_14":dict(technology="merfish",pixels=100,label="FUSEmap_main_level",radius=100,
    parameters=fixed(.20,100,.25,60,6),basis="historical final smallest-pair template; source-unit calibration assumption retained"),
}
SELECTED_CASES = list(CASES)
# These are protocol choices, fixed BEFORE seeing sweep results. Start a new OUT if changed.
protocol=dict(ratios=RATIOS,seeds=SEEDS,reference_ratio=REFERENCE_RATIO,cases={n:CASES[n] for n in SELECTED_CASES},
    min_points=MIN_POINTS,max_points=MAX_POINTS,distance_step_um=25,threads=THREADS,timeout_seconds=TIMEOUT_SECONDS,
    source_sha256={str(p.relative_to(SRC)):sha_file(p) for p in sorted((SRC/"raftup").rglob("*.py"))},
    python=sys.version,platform=platform.platform(),package_versions={"numpy":np.__version__,"torch":torch.__version__})
protocol_path=OUT/"protocol.json"
if REUSE_EXISTING_RESULTS:
    archived_protocol=json.loads(protocol_path.read_text())
    assert archived_protocol["ratios"]==RATIOS and archived_protocol["seeds"]==SEEDS
    for name in SELECTED_CASES:
        assert archived_protocol["cases"][name]==CASES[name], "Archived parameters differ"
    print("Reading archived runs; current source is not used to recompute results.")
elif protocol_path.exists():
    assert json.loads(protocol_path.read_text())==protocol, "Protocol/source changed: use a new OUT directory."
else: write_json(protocol_path,protocol)
protocol_sha=sha_file(protocol_path)
display(pd.DataFrame([dict(case=n,**c["parameters"],technology=c["technology"],basis=c["basis"]) for n,c in CASES.items()]))
print("Alignment jobs: 1 (Deep-STARmap 10%); other 17 settings use archived results.")


## 1. Freeze input slices and feature costs once

Only coordinates/cell IDs and evaluation labels are cached in lightweight slices. Labels
are stripped before alignment. Existing feature costs are reused and fingerprinted; no
cost retraining occurs per ratio or seed. The old Deep-STARmap cache records slice names,
shape and training parameters but not a full cell-order hash; its provenance limitation
is retained in the manifest. The study locks the current ordered input and cost hashes.

Spatial cutoffs are micrometers for calibrated DLPFC/MERFISH. Deep-STARmap retains the
previous `pixels_per_100um=100` numerical convention; physical calibration is unverified.
GPR radii remain in ORIGINAL input units, fixed within each case.


In [ ]:
if not REUSE_EXISTING_RESULTS:
    def load_case(name, dest):
        generated_cost_seconds=0.0
        if name=="dlpfc_0708":
            a=_load_data.load_DLPFC(root_dir=str(REPRO/"1_DLPFC_all_exp/DLPFC12"),section_id="151507")
            b=_load_data.load_DLPFC(root_dir=str(REPRO/"1_DLPFC_all_exp/DLPFC12"),section_id="151508")
            cp=REPRO/"0_preprocessing/feature_cost_in_DLPFC/cost_mac_reproducible_robust_somde/151507_151508_somde_cost_matrix.pt"
        elif name=="merfish_0409":
            folder=ROOT/"cutoff_automation_test/merfish_automation/slice_cache"
            a=ad.read_h5ad(folder/"slice_-0.04.h5ad"); b=ad.read_h5ad(folder/"slice_-0.09.h5ad")
            cp=REPRO/"0_preprocessing/feature_cost_in_MERFISH/cost_mac_reproducible/-0.04_-0.09_somde_cost_matrix.pt"
        else:
            folder=ROOT/"2_3d_reconstruction_exp/Deep-STARmap/processed/synthetic_10um_random_z_rotation_seed42"
            a=ad.read_h5ad(folder/"section_13.h5ad");b=ad.read_h5ad(folder/"section_14.h5ad")
            cp=DEEP_COST
            if not cp.exists():
                if not COMPUTE_MISSING_DEEP_COST: raise FileNotFoundError(f"Set DEEP_COST or explicitly enable one-time computation: {cp}")
                cp=dest/"generated_scanit_cost.npy"
                aa,bb=a.copy(),b.copy()
                for obj in (aa,bb): obj.obs=obj.obs.iloc[:,:0].copy(); obj.uns.clear()
                started=time.perf_counter()
                cost=cs.compute_scanit_cost(aa,bb,technology="merfish",n_epoch=3500,n_h=32,seed=0,device="cpu")
                generated_cost_seconds=time.perf_counter()-started
                np.save(cp,cost); del cost,aa,bb
        assert a.obs_names.is_unique and b.obs_names.is_unique
        cost = (cs._as_numpy(torch.load(cp,map_location="cpu",weights_only=False)) if cp.suffix==".pt" else np.load(cp,mmap_mode="r"))
        assert cost.shape==(len(a),len(b)) and np.isfinite(cost).all() and np.all(cost>=0)
        return a,b,cost,cp,generated_cost_seconds
    
    prepared={}
    for name in SELECTED_CASES:
        if name != SUPPLEMENT_CASE:
            prepared[name]=json.loads((ARCHIVE_OUT/"inputs"/name/"manifest.json").read_text())
            continue
        folder=OUT/"inputs"/name;folder.mkdir(parents=True,exist_ok=True)
        manifest_path=folder/"manifest.json"
        if manifest_path.exists():
            manifest=json.loads(manifest_path.read_text())
            assert manifest["protocol_sha256"]==protocol_sha
            for filename,digest in manifest["files"].items(): assert sha_file(folder/filename)==digest,filename
        else:
            started=time.perf_counter()
            a,b,cost,cp,cost_seconds=load_case(name,folder)
            for side,obj in [("A",a),("B",b)]:
                key=CASES[name]["label"];assert key in obj.obs,key
                slim=ad.AnnData(sparse.csr_matrix((len(obj),1)),obs=obj.obs[[key]].copy())
                slim.obsm["spatial"]=np.asarray(obj.obsm["spatial"],float)[:,:2]
                slim.write_h5ad(folder/(side+".h5ad"))
            np.save(folder/"feature_cost.npy",np.asarray(cost))
            manifest=dict(protocol_sha256=protocol_sha,shape=list(cost.shape),cost_source=str(cp),source_cost_sha256=sha_file(cp),
                cost_generation_seconds=cost_seconds,preparation_seconds=time.perf_counter()-started,
                cost_order_assumption="Reuse the existing pair cost in original cell order; source cache lacks an independent cell-order signature" if name=="deep_starmap_13_14" else "Existing ordered pair inputs/cost",
                files={f:sha_file(folder/f) for f in ["A.h5ad","B.h5ad","feature_cost.npy"]})
            write_json(manifest_path,manifest)
            del a,b,cost;gc.collect()
        old_manifest=json.loads((ARCHIVE_OUT/"inputs"/name/"manifest.json").read_text())
        assert manifest["shape"] == old_manifest["shape"]
        assert manifest["source_cost_sha256"] == old_manifest["source_cost_sha256"], "Feature cost changed"
        # Compare semantic input identity; H5AD byte hashes may vary when rewritten.
        for side in ("A", "B"):
            old_slice=ad.read_h5ad(ARCHIVE_OUT/"inputs"/name/(side+".h5ad"))
            new_slice=ad.read_h5ad(folder/(side+".h5ad"))
            assert old_slice.obs_names.equals(new_slice.obs_names), "Cell order changed"
            np.testing.assert_array_equal(old_slice.obsm["spatial"], new_slice.obsm["spatial"])
            assert old_slice.obs.equals(new_slice.obs), "Evaluation labels changed"
        prepared[name]=manifest
    rows=[]
    for name in SELECTED_CASES:
        na,nb=prepared[name]["shape"];c=CASES[name]
        for ratio in RATIOS:
            counts=([round(ratio*min(na,nb))]*2 if c["technology"]=="merfish" else [int(np.ceil(ratio*na)),int(np.ceil(ratio*nb))])
            feasible=min(counts)>=MIN_POINTS and (MAX_POINTS is None or max(counts)<=MAX_POINTS) and counts[0]>=c["parameters"]["k1"] and counts[1]>=c["parameters"]["k2"]
            rows.append(dict(case=name,ratio=ratio,n_A=na,n_B=nb,target_A=counts[0],target_B=counts[1],preflight_feasible=feasible))
    plan=pd.DataFrame(rows);plan.to_csv(OUT/"sampling_plan.csv",index=False);display(plan)
else:
    print("Archived analysis mode: no experiment execution.")

## 2. Isolated worker and evaluation

Each run uses a fresh sequential process, fixed CPU thread counts and the same feature
cost. Alignment time excludes input loading, scoring and saving. A small fixed kernel
warm-up is excluded and timed separately; this prevents the first ratio paying all JIT
startup costs. Cost generation/preparation is recorded separately. No parallel timing
comparisons are made. The randomized execution order reduces systematic order effects.

LAA uses covered-row argmax labels (invalid labels excluded), alongside coverage so that
high accuracy on very few matches cannot hide missing matches. Sparse GPR implements the
same formula as `GPR_original`, verified below on a small matrix. No GT is read by the
alignment call. Saved hard matches use -1 for unmatched rows.


In [ ]:
if not REUSE_EXISTING_RESULTS:
    WORKER_SOURCE = 'import os,sys,json,time,traceback,contextlib,gc\nfrom pathlib import Path\nos.environ.setdefault("NUMBA_CACHE_DIR", "/private/tmp/raftup_cutoff_numba" if sys.platform=="darwin" else "/tmp/raftup_cutoff_numba")\nos.environ.setdefault("MPLCONFIGDIR", "/private/tmp/raftup_cutoff_mpl" if sys.platform=="darwin" else "/tmp/raftup_cutoff_mpl")\njob=json.loads(Path(sys.argv[1]).read_text())\nos.environ["OMP_NUM_THREADS"]=str(job["threads"])\nos.environ["MKL_NUM_THREADS"]=str(job["threads"])\nos.environ["OPENBLAS_NUM_THREADS"]=str(job["threads"])\nsys.path.insert(0,job["src"])\nimport numpy as np\nimport pandas as pd\nimport anndata as ad\nimport torch\nfrom scipy import sparse\nfrom scipy.spatial import cKDTree\nfrom scipy.spatial.distance import cdist\nfrom raftup import _cutoff_selection as cs\nfrom raftup._cutoff_selection import run_manual_alignment\n\ndef gpr_sparse(P,x,y,radius,p_cut=1e-19):\n    valid=np.asarray(P.sum(1)).ravel()>=p_cut\n    if not valid.any():return float("nan")\n    P=P[valid].copy();P.data[P.data<1e-100]=0;P.eliminate_zeros()\n    mass=np.asarray(P.sum(1)).ravel();W=sparse.diags(1/np.where(mass>0,mass,1))@P\n    def neighbors(z):\n        lists=cKDTree(z).query_ball_point(z,radius)\n        rr=np.repeat(np.arange(len(z)),[len(v) for v in lists]);cc=np.concatenate(lists)\n        return sparse.csr_matrix((np.ones(len(rr)),(rr,cc)),shape=(len(z),len(z)))\n    NA=neighbors(x[valid]);NB=neighbors(y)\n    NA=sparse.diags(1/np.asarray(NA.sum(1)).ravel())@NA\n    total=0.\n    for start in range(0,len(mass),128):\n        stop=min(start+128,len(mass))\n        total+=float(((NA[start:stop]@W)@NB).multiply(W[start:stop]).sum())\n    return total/len(mass)\n\ndef evaluate(P,a,b,label,radius,case):\n    ca=np.asarray(P.sum(1)).ravel()>0;cb=np.asarray(P.sum(0)).ravel()>0\n    pred=np.full(len(a),-1,dtype=np.int64)\n    pred[ca]=np.asarray(P[ca].argmax(1)).ravel()\n    la=a.obs[label].astype(str).to_numpy();lb=b.obs[label].astype(str).to_numpy()\n    rows=np.flatnonzero(ca);cols=pred[ca]\n    valid=~np.isin(la[rows],["-1","nan","None"]) & ~np.isin(lb[cols],["-1","nan","None"])\n    if case=="merfish_0409":\n        allowed=["BST","MPA","MPN","PV","PVH","PVT","V3","fx"]\n        valid &= np.isin(la[rows],allowed)&np.isin(lb[cols],allowed)\n    laa=float(np.mean(la[rows[valid]]==lb[cols[valid]])) if valid.any() else float("nan")\n    return dict(LAA=laa,GPR=gpr_sparse(P,a.obsm["spatial"],b.obsm["spatial"],radius),\n        coverage_A=float(ca.mean()),coverage_B=float(cb.mean()),valid_label_rows=int(valid.sum()),nnz=int(P.nnz)),pred,ca,cb\n\ndef warmup():\n    # Exercise the same graph/transport/recovery kernels without using study data.\n    z=np.array([[10.,10.],[40.,10.],[10.,40.],[40.,40.],[25.,25.],[60.,35.]])\n    d=cdist(z,z);m=d/max(d.max(),1)\n    cs._fsgw_utils_test_sparse.fsgw_mvc_exp_sparse_heap(d,d,m,gw_cutoff=10,w_cutoff=.2,seed=42,show_progress=False)\n    cs._recoverfull_new_new_knn.recover_full_mapping_knn(M=m,X1=z,X2=z,P=np.eye(6)/6,\n        idx1=np.arange(6),idx2=np.arange(6),k1=2,k2=2,thresh_CGW=100,thresh_CCC=.5,eps=.01)\n\nif __name__=="__main__":\n    torch.set_num_threads(job["threads"])\n    folder=Path(job["folder"]);folder.mkdir(parents=True,exist_ok=True)\n    result={k:job[k] for k in ["case","ratio","seed","protocol_sha256"]}\n    stage="loading";wall=time.perf_counter()\n    try:\n        inp=Path(job["input"]);a=ad.read_h5ad(inp/"A.h5ad");b=ad.read_h5ad(inp/"B.h5ad")\n        cost=np.load(inp/"feature_cost.npy",mmap_mode="r")\n        aa,bb=a.copy(),b.copy()\n        for obj in (aa,bb):obj.obs=obj.obs.iloc[:,:0].copy();obj.uns.clear()\n        stage="warmup";t=time.perf_counter();warmup();result["warmup_seconds"]=time.perf_counter()-t\n        stage="alignment";t=time.perf_counter()\n        p,d=run_manual_alignment(aa,bb,cost,technology=job["technology"],pixels_per_100um=job["pixels"],\n            downsampling_ratio=job["ratio"],random_state=job["seed"],cost_method="provided",\n            min_downsampled_points=job["min_points"],max_downsampled_points=job["max_points"],\n            distance_step_um=25,output_dir=folder/"artifacts",**job["parameters"])\n        result["alignment_wall_seconds"]=time.perf_counter()-t # includes manual artifact writes\n        result["alignment_seconds"]=d["total_seconds"] # excludes writes; principal timing\n        for key in ["cost_seconds","sampling_seconds","step1_seconds","step2_seconds"]:result[key]=d[key]\n        for key in ["ratio_a","ratio_b"]:result[key]=d["sampling"][key]\n        result["sample_A"],result["sample_B"]=d["sampling"]["actual_counts"]\n        result["unique_sample_A"],result["unique_sample_B"]=d["sampling"]["unique_counts"]\n        result["sample_cost_max"]=float(np.max(cost[np.ix_(d["sampling"]["indices_a"],d["sampling"]["indices_b"])]))\n        P=sparse.csr_matrix(p);del p;gc.collect()\n        sparse.save_npz(folder/"alignment_matrix.npz",P)\n        (folder/"artifacts/alignment_matrix.npy").unlink() # only our new redundant dense output\n        stage="evaluation";t=time.perf_counter()\n        scores,pred,ca,cb=evaluate(P,a,b,job["label"],job["radius"],job["case"])\n        result.update(scores);result["evaluation_seconds"]=time.perf_counter()-t\n        np.savez_compressed(folder/"hard_matches.npz",prediction=pred,coverage_A=ca,coverage_B=cb,\n            sample_A=np.array(d["sampling"]["indices_a"]),sample_B=np.array(d["sampling"]["indices_b"]))\n        result["status"]="ok" if P.nnz else "empty_alignment"\n    except Exception as exc:\n        result.update(status="failed",stage=stage,error=str(exc))\n        (folder/"traceback.txt").write_text(traceback.format_exc())\n    result["worker_wall_seconds"]=time.perf_counter()-wall\n    result={k:(None if isinstance(v,float) and not np.isfinite(v) else v) for k,v in result.items()}\n    tmp=folder/"result.json.tmp";tmp.write_text(json.dumps(result,indent=2,allow_nan=False));tmp.replace(folder/"result.json")\n    print(json.dumps(result),flush=True)\n'
    WORKER = OUT / "sensitivity_worker.py"
    if WORKER.exists(): assert WORKER.read_text()==WORKER_SOURCE, "Worker changed: select new OUT"
    else: WORKER.write_text(WORKER_SOURCE)
    # Verify sparse GPR against the existing implementation (no experiment execution).
    import ast
    from raftup import _metrics_two_gpr
    node=next(x for x in ast.parse(WORKER_SOURCE).body if isinstance(x,ast.FunctionDef) and x.name=="gpr_sparse")
    exec(compile(ast.Module(body=[node],type_ignores=[]),"gpr_check","exec"))
    rng=np.random.default_rng(81); test_p=rng.random((9,7)); test_p[test_p<.5]=0; test_p[0]=0
    x=rng.random((9,2));y=rng.random((7,2))
    np.testing.assert_allclose(gpr_sparse(sparse.csr_matrix(test_p),x,y,.4),
        _metrics_two_gpr.GPR_original(test_p.copy(),x,y,.4,1e-19),rtol=1e-12,atol=1e-12)
    print("Sparse GPR formula check passed")
else:
    print("Archived analysis mode: no experiment execution.")

## 3. Run/resume only Deep-STARmap 10%

This cell schedules exactly one setting. All other results are read from the original archive. It does not rerun the other 17 experiments.

In [ ]:
if not REUSE_EXISTING_RESULTS:
    jobs=[]
    for name,ratio,seed in itertools.product([SUPPLEMENT_CASE],[SUPPLEMENT_RATIO],SEEDS):
        folder=OUT/"runs"/name/f"ratio_{ratio:.4f}"/f"seed_{seed}"
        job=dict(case=name,ratio=ratio,seed=seed,folder=str(folder),input=str(OUT/"inputs"/name),src=str(SRC),
            protocol_sha256=protocol_sha,threads=THREADS,min_points=MIN_POINTS,max_points=MAX_POINTS,
            **{k:CASES[name][k] for k in ["technology","pixels","label","radius","parameters"]})
        jobs.append(job)
    order=np.random.default_rng(2026).permutation(len(jobs))
    write_json(OUT/"execution_order.json",[jobs[i] for i in order])
    all_results=[]
    for number,i in enumerate(order,1):
        assert protocol["source_sha256"]=={str(p.relative_to(SRC)):sha_file(p) for p in sorted((SRC/"raftup").rglob("*.py"))}, "Source changed during sweep; stop and use a new OUT"
        job=jobs[i];folder=Path(job["folder"]);folder.mkdir(parents=True,exist_ok=True)
        result_path=folder/"result.json"
        old=json.loads(result_path.read_text()) if result_path.exists() else None
        if old is not None: assert old["protocol_sha256"]==protocol_sha
        if old is not None and (old["status"]=="ok" or not RETRY_FAILED):
            row=old
        else:
            print(f"[{number}/{len(jobs)}] {job['case']} ratio={job['ratio']} seed={job['seed']}",flush=True)
            candidate=plan[(plan.case==job["case"]) & (plan.ratio==job["ratio"])].iloc[0]
            if not candidate.preflight_feasible:
                row={k:job[k] for k in ["case","ratio","seed","protocol_sha256"]}
                row.update(status="infeasible",error="Requested count conflicts with fixed k or point bounds")
            else:
                write_json(folder/"job.json",job);started=time.perf_counter()
                with (folder/"worker.log").open("w") as log:
                    try:
                        proc=subprocess.run([PYTHON,str(WORKER),str(folder/"job.json")],stdout=log,stderr=subprocess.STDOUT,timeout=TIMEOUT_SECONDS)
                        if proc.returncode==0 and result_path.exists(): row=json.loads(result_path.read_text())
                        else: row=dict(case=job["case"],ratio=job["ratio"],seed=job["seed"],protocol_sha256=protocol_sha,status="process_failed",error=f"exit={proc.returncode}; see worker.log")
                    except subprocess.TimeoutExpired:
                        row=dict(case=job["case"],ratio=job["ratio"],seed=job["seed"],protocol_sha256=protocol_sha,status="timeout",error=f"Exceeded {TIMEOUT_SECONDS}s")
                row["process_wall_seconds"]=time.perf_counter()-started
            write_json(result_path,row)
        all_results.append(row)
        pd.DataFrame(all_results).to_csv(OUT/"per_run_metrics.csv",index=False)
    print("Finished requested sweep. Failed runs remain in the results.")
else:
    print("Archived analysis mode: no experiment execution.")

## 4. Single-run scores and sensitivity of correspondences

No mean, SD or confidence interval is computed from a single run. Failures are retained.
Compare hard-match Jaccard, agreement among jointly covered rows, coverage-set Jaccard
and sampled-point Jaccard between each ratio and 5%, and between adjacent ratios.
These are density-sensitivity diagnostics; high consistency can still reflect consistently
incorrect alignments, so interpret them alongside LAA, GPR and both coverages.


In [ ]:
# Reload results without repeating alignment.
records=[]
result_folders={}
for name, ratio in itertools.product(SELECTED_CASES, RATIOS):
    use_new = not REUSE_EXISTING_RESULTS and name == SUPPLEMENT_CASE and np.isclose(ratio, SUPPLEMENT_RATIO)
    source = OUT if use_new else ARCHIVE_OUT
    folder=source/"runs"/name/f"ratio_{ratio:.4f}"/f"seed_{FIXED_SEED}"
    result_folders[(name, ratio)]=folder
    path=folder/"result.json"
    if path.exists():
        row=json.loads(path.read_text())
        row["result_source"]=str(path)
        row["source_kind"]="supplement_no_cap" if use_new else "archived_800_cap"
        records.append(row)
    elif not use_new:
        raise FileNotFoundError(f"Missing archived result: {path}")
df=pd.DataFrame(records)
df=df[df.case.isin(SELECTED_CASES)].copy()
assert len(df), "Run the sweep first"
assert df.seed.eq(FIXED_SEED).all(), "Use the single-seed output directory"
assert not df.duplicated(["case","ratio"]).any()
metrics=["LAA","GPR","coverage_A","coverage_B","alignment_seconds","sampling_seconds","step1_seconds","step2_seconds","ratio_a","ratio_b"]
for col in metrics:
    if col not in df: df[col]=np.nan
# Include not-yet-run settings so interrupted sweeps remain visible.
summary=pd.DataFrame(itertools.product(SELECTED_CASES,RATIOS),columns=["case","ratio"]).merge(df,on=["case","ratio"],how="left")
summary["status"]=summary.status.fillna("not_run")
summary.to_csv(REPORT_OUT/"single_run_summary.csv",index=False)

def jaccard(a,b):
    a,b=set(a),set(b);return len(a&b)/len(a|b) if a|b else np.nan

def load_matches(name,ratio):
    path=result_folders[(name,ratio)]/"hard_matches.npz"
    with np.load(path) as f:return {k:f[k] for k in f.files}

comparisons=[]
for name in SELECTED_CASES:
    comparisons_to_make=[("vs_5pct",REFERENCE_RATIO,r) for r in RATIOS if not np.isclose(r,REFERENCE_RATIO)]
    comparisons_to_make += [("adjacent_ratios",r1,r2) for r1,r2 in zip(sorted(RATIOS)[:-1],sorted(RATIOS)[1:])]
    for kind,r1,r2 in comparisons_to_make:
        left=summary[(summary.case==name)&np.isclose(summary.ratio,r1)].iloc[0]
        right=summary[(summary.case==name)&np.isclose(summary.ratio,r2)].iloc[0]
        row=dict(case=name,comparison=kind,ratio_from=r1,ratio_to=r2,status_from=left.status,status_to=right.status)
        if left.status=="ok" and right.status=="ok":
            a,b=load_matches(name,r1),load_matches(name,r2)
            p,q=a["prediction"],b["prediction"];pa=p>=0;pb=q>=0
            same=(p==q)&pa&pb;union=int(pa.sum()+pb.sum()-same.sum());common=pa&pb
            row.update(hard_match_jaccard=float(same.sum()/union) if union else np.nan,
                common_covered_agreement=float(same[common].mean()) if common.any() else np.nan,
                coverage_A_jaccard=jaccard(np.flatnonzero(pa),np.flatnonzero(pb)),
                coverage_B_jaccard=jaccard(np.flatnonzero(a["coverage_B"]),np.flatnonzero(b["coverage_B"])),
                sample_A_jaccard=jaccard(a["sample_A"],b["sample_A"]),sample_B_jaccard=jaccard(a["sample_B"],b["sample_B"]),
                LAA_delta=right.LAA-left.LAA,GPR_delta=right.GPR-left.GPR,
                runtime_ratio=right.alignment_seconds/left.alignment_seconds if left.alignment_seconds>0 else np.nan)
        comparisons.append(row)
changes=pd.DataFrame(comparisons)
for col in ["hard_match_jaccard","common_covered_agreement","LAA_delta","GPR_delta","runtime_ratio"]:
    if col not in changes:changes[col]=np.nan
changes.to_csv(REPORT_OUT/"between_ratio_consistency.csv",index=False)
display(summary[["case","ratio","status",*metrics]])
display(changes)


## 5. Figures and descriptive report

Each point is one run, without error bars. Failed settings have no imputed score or time.
Matching consistency is shown relative to 5%; CSV also includes adjacent-ratio comparisons.
The baseline is shown as 1 only if that baseline run succeeded. Runtime excludes cost
preparation, kernel warm-up, evaluation and output writes. Step-1 normalization still uses
the sampled maximum cost; its recorded denominator can change with ratio.


Purple dashed lines mark the final requested ratio from the original automatic workflow: DLPFC 0708 = 5%, MERFISH 0409 = 5%, Deep-STARmap 13–14 = 6.7%. These lines mark ratio locations only; the fixed-parameter manual sweep does not rerun automatic selection. Matching consistency remains relative to the common 5% reference.

Run status: `ok` = successful run; `infeasible` = requested sample count conflicts with fixed point bounds or k; `failed` = an exception; `empty_alignment` = no matches; `timeout` = time limit exceeded; `process_failed` = worker process failed; `not_run` = no saved result. Status is not an accuracy score.

In [ ]:
import matplotlib.pyplot as plt
FIG=REPORT_OUT/"figures";FIG.mkdir(exist_ok=True)
colors=plt.get_cmap("tab10").colors
CASE_LABELS = {"dlpfc_0708": "DLPFC_0708", "merfish_0409": "MERFISH_0409", "deep_starmap_13_14": "Deep-STARmap_1314"}
# Final target ratios recorded in the original automatic workflow decisions.
AUTO_BASELINE_RATIOS = {"dlpfc_0708": 0.05, "merfish_0409": 0.05, "deep_starmap_13_14": 0.067}

def mark_auto_baseline(ax, name, annotate=True, fontsize=8):
    ratio = AUTO_BASELINE_RATIOS[name] * 100
    ax.axvline(ratio, color="#6b21a8", linestyle="--", linewidth=1.5,
               alpha=.85, zorder=0, label="default ratio")
    if annotate:
        ax.legend(loc="best", fontsize=fontsize, framealpha=.85)

def savefig(fig,name):
    fig.savefig(FIG/(name+".png"),dpi=200,bbox_inches="tight")
    fig.savefig(FIG/(name+".pdf"),bbox_inches="tight")
    plt.show();plt.close(fig)

panels=[("LAA","LAA"),("GPR","GPR"),("coverage_A","Coverage A"),("coverage_B","Coverage B"),("alignment_seconds","runtime (in seconds)")]
fig,axes=plt.subplots(len(SELECTED_CASES),len(panels),figsize=(20,3*len(SELECTED_CASES)),squeeze=False)
for i,name in enumerate(SELECTED_CASES):
    sub=summary[summary.case==name].sort_values("ratio").copy()
    for j,(metric,title) in enumerate(panels):
        ax=axes[i,j];values=sub[metric].where(sub.status=="ok")
        ax.plot(sub.ratio*100,values,marker="o",color=colors[i])
        ax.set_title(title, fontsize=14);ax.set_xlabel("Downsampling ratio (%)", fontsize=14);ax.grid(alpha=.2)
        if j==0:ax.set_ylabel(CASE_LABELS[name], fontsize=14)
        if metric!="alignment_seconds":ax.set_ylim(-.03,1.03)
        ax.set_xticks(np.array(RATIOS)*100)
        mark_auto_baseline(ax, name, fontsize=12)
fig.suptitle("Sensitivity analysis of RAFT-UP under different downsampling ratios", fontsize=22, y=1.02)
fig.tight_layout();savefig(fig,"accuracy_runtime")
fig,axes=plt.subplots(len(SELECTED_CASES),3,figsize=(14,3*len(SELECTED_CASES)),squeeze=False)
for i,name in enumerate(SELECTED_CASES):
    sub=summary[summary.case==name].sort_values("ratio")
    c=changes[(changes.case==name)&(changes.comparison=="vs_5pct")].copy()
    baseline=sub[np.isclose(sub.ratio,REFERENCE_RATIO)].iloc[0]
    for metric in ["hard_match_jaccard","common_covered_agreement"]:
        x=c.ratio_to.to_numpy();y=c[metric].to_numpy()
        if baseline.status=="ok":x=np.r_[x,REFERENCE_RATIO];y=np.r_[y,1.]
        order=np.argsort(x);axes[i,0].plot(x[order]*100,y[order],"o-",label=metric)
    axes[i,0].set_ylabel(CASE_LABELS[name]);axes[i,0].set_title("Matching consistency vs 5%")
    for side in ["a","b"]:axes[i,1].plot(sub.ratio*100,sub["ratio_"+side]*100,"o-",label="Actual "+side.upper())
    axes[i,1].plot(sub.ratio*100,sub.ratio*100,"k--",label="Downsampling ratio");axes[i,1].set_title("Achieved sampling (%)")
    axes[i,2].scatter(sub.ratio*100,sub.status.eq("ok").astype(int))
    for r in sub.itertuples():axes[i,2].annotate(r.status,(r.ratio*100,int(r.status=="ok")),rotation=25,fontsize=7)
    axes[i,2].set_title("Run status")
    for j in [0,2]:axes[i,j].set_ylim(-.08,1.25)
    for ax in axes[i]:
        ax.set_xlabel("Downsampling ratio (%)");ax.grid(alpha=.2)
        mark_auto_baseline(ax, name)
    for j in [0,1]:axes[i,j].legend(fontsize=7)
fig.tight_layout();savefig(fig,"consistency_sampling_status")
fig,axes=plt.subplots(1,len(SELECTED_CASES),figsize=(20,4),squeeze=False)
for i,name in enumerate(SELECTED_CASES):
    sub=summary[summary.case==name].sort_values("ratio")
    for m in ["sampling_seconds","step1_seconds","step2_seconds"]:axes[0,i].plot(sub.ratio*100,sub[m].where(sub.status=="ok"),"o-",label=m)
    axes[0,i].set_title(CASE_LABELS[name]);axes[0,i].set_xlabel("Downsampling ratio (%)");axes[0,i].set_ylabel("Seconds");axes[0,i].legend(fontsize=7, loc="center left")
    mark_auto_baseline(axes[0,i], name)
fig.tight_layout();savefig(fig,"stage_runtime")

lines=["# Downsampling sensitivity: fixed seed", "",f"Recorded {len(df)} of {len(SELECTED_CASES)*len(RATIOS)} settings; seed={FIXED_SEED}.",
    "Each score/time is from one run. No uncertainty or stochastic stability is estimated.","",
    "| Case | Ratio | Status | LAA | GPR | Coverage A | Coverage B | Alignment s |",
    "|---|---:|---|---:|---:|---:|---:|---:|"]
for r in summary.to_dict("records"):
    def value(m):return f"{r[m]:.4f}" if r["status"]=="ok" and pd.notna(r[m]) else "NA"
    lines.append(f"| {r['case']} | {100*r['ratio']:g}% | {r['status']} | "+" | ".join(value(m) for m in ["LAA","GPR","coverage_A","coverage_B","alignment_seconds"])+" |")
lines += ["", "## Interpretation", "",
    "- The merged study uses 17 archived runs and one supplemental Deep-STARmap 10% run; source paths and original protocol hashes are retained in the summary CSV.",
    "- Correspondence agreement measures sensitivity to ratio changes, not random-seed stability or accuracy.",
    "- DLPFC grid is deterministic; MERFISH maxmin initialization and solver seed are held fixed.",
    "- Fixed cost excludes feature-learning variability. The sampled cost normalization denominator may change with ratio.",
    "- Interpret LAA/GPR jointly with coverage and failures. One pair per family is descriptive, not dataset-level evidence.",
    "- Archived Deep-STARmap 10% was skipped under the old 800-point bound. New runs allow it with a runtime notice. Physical XY calibration remains unverified.",
    "- Reported alignment runtime excludes one-time preparation, warm-up, evaluation and writes."]
(REPORT_OUT/"report.md").write_text("\n".join(lines))
print("Results:",REPORT_OUT)
